# Notebook de Ingestão
Objetivo: importar arquivos CSV para DuckDB e armazenar em `bronze_produtos`.

Este notebook realiza as etapas:
1. Importar bibliotecas
2. Conectar ao DuckDB (arquivo local)
3. Ler arquivos CSV da pasta `landing/`
4. Criar a tabela `bronze_produtos` se necessário
5. Inserir os dados lidos na tabela
6. Validar a ingestão e mostrar os primeiros registros

Observação: execute cada célula em ordem; para evitar perda de `df`, faça o `INSERT` logo após cada leitura ou concatene os DataFrames antes de inserir.

In [5]:
# Verifica a versão do DuckDB instalada (útil para debug)
import duckdb
print(duckdb.__version__)


1.5.3


In [6]:
# Import das bibliotecas principais: pandas para manipulação, os para caminhos e datetime para marcação
import pandas as pd
import os
from datetime import datetime


In [7]:
con = duckdb.connect(database='dados_duckbd.db', read_only=False)       

In [8]:
# Leitura do arquivo `z0019_1.csv` usando separador `;`.
# Adicionamos `nome_arquivo` e `data_ingestao` para rastreabilidade.
arquivo = 'z0019_1.csv'
data_ingestao = datetime.now()
df = pd.read_csv(f'../landing/{arquivo}', sep=';')
df['nome_arquivo'] = arquivo
df['data_ingestao'] = data_ingestao
df.head()


,NATB,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,1001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-06-06 22:26:16.802236
1,1002,MARTELO,BT50,100,1500,z0019_1.csv,2026-06-06 22:26:16.802236
2,1003,PREGO,BT50,100,50,z0019_1.csv,2026-06-06 22:26:16.802236


In [9]:
# Leitura do arquivo `z0019_2.csv` usando separador `;`.
# Atenção: se você executar esta célula sem inserir o `df` anterior, o objeto `df` será sobrescrito.
# Execute o `INSERT` após cada leitura ou concatene os DataFrames antes de inserir na tabela.
arquivo = 'z0019_2.csv'
data_ingestao = datetime.now()
df = pd.read_csv(f'../landing/{arquivo}', sep=';')
df['nome_arquivo'] = arquivo
df['data_ingestao'] = data_ingestao
df.head()


,NATB,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,1003,SERRA,BT50,100,200,z0019_2.csv,2026-06-06 22:26:16.890615
1,1003,MACHADO,BT50,100,100,z0019_2.csv,2026-06-06 22:26:16.890615
2,1003,PREGO,BT50,100,60,z0019_2.csv,2026-06-06 22:26:16.890615


In [10]:
# Exibe timestamp atual (útil para logs e comparação de `data_ingestao`)
print(datetime.now())


2026-06-06 22:26:16.918645


In [11]:
# Cria a tabela `bronze_produtos` caso não exista. Ajuste tipos se necessário.
con.execute("""
            CREATE TABLE IF NOT EXISTS bronze_produtos(
            NATBR VARCHAR,
            MAKTX VARCHAR,
            WERKS VARCHAR,
            MAINS VARCHAR,
            LABST VARCHAR,
            nome_arquivo VARCHAR,
            data_ingestao TIMESTAMP
              ) 
            """)


In [12]:
resultado = con.execute("SELECT * FROM bronze_produtos").fetchdf()
print(resultado)

Empty DataFrame
Columns: [NATBR, MAKTX, WERKS, MAINS, LABST, nome_arquivo, data_ingestao]
Index: []


In [13]:
con.execute("SHOW TABLES").fetchdf()

,name
0,bronze_produtos
1,bronze_z0019
